# Day 14 — Caching & Persistence

**What you will learn:**
- `cache()` vs `persist()` — what each does and when to use them
- Storage levels: `MEMORY_ONLY`, `MEMORY_AND_DISK`, `DISK_ONLY`, `OFF_HEAP`
- `unpersist()` — releasing cached data
- When caching helps vs when it hurts
- Verifying cache with `explain()` and Spark UI

**Exam relevance:** Optimisation (10%) and Architecture (20%) — storage levels, when caching pays off, and the cost of recomputation vs materialisation.

In [ ]:
from pyspark.sql.functions import col, avg, count, sum, round
from pyspark import StorageLevel

data = [
    ("Alice",   "Engineering", "RO", 95000, 2022),
    ("Bob",     "Marketing",   "RO", 72000, 2021),
    ("Carol",   "Engineering", "UK", 110000, 2020),
    ("Dave",    "Sales",       "DE", 58000,  2023),
    ("Eve",     "Marketing",   "RO", 81000,  2021),
    ("Frank",   "Engineering", "DE", 88000,  2022),
    ("Grace",   "Sales",       "RO", 61000,  2020),
    ("Hank",    "Engineering", "UK", 102000, 2023),
    ("Iris",    "Marketing",   "DE", 76000,  2022),
    ("Jack",    "Sales",       "UK", 67000,  2021),
]
df = spark.createDataFrame(data, ["name", "dept", "country", "salary", "hire_year"])

## 1. cache() — Shortcut for MEMORY_AND_DISK

`cache()` is equivalent to `persist(StorageLevel.MEMORY_AND_DISK)`.

**Key rule:** Caching is **lazy** — the DataFrame is only materialised when an action is triggered.

In [ ]:
# Simulate an expensive transformation (join + filter)
from pyspark.sql.functions import broadcast

dept_info = spark.createDataFrame([
    ("Engineering", "Tech"),
    ("Marketing",   "Ops"),
    ("Sales",       "Ops"),
], ["dept", "division"])

enriched = df.join(broadcast(dept_info), on="dept") \
             .filter(col("salary") > 60000)

# Cache the result — expensive join + filter won't repeat on subsequent actions
enriched.cache()

# First action — triggers materialisation
print("Row count:", enriched.count())

# Second action — reads from cache, no re-computation
enriched.groupBy("dept").agg(round(avg("salary"), 0).alias("avg_salary")).show()

## 2. persist() — Choose Your Storage Level

| Storage Level | Memory | Disk | Serialized | Replicas | When to use |
|---|---|---|---|---|---|
| `MEMORY_ONLY` | ✅ | ❌ | No | 1 | Fits in RAM, recompute is cheap if evicted |
| `MEMORY_AND_DISK` | ✅ | ✅ | No | 1 | Default `cache()` — spills to disk if RAM full |
| `MEMORY_ONLY_SER` | ✅ | ❌ | Yes | 1 | RAM limited — serialization saves space |
| `MEMORY_AND_DISK_SER` | ✅ | ✅ | Yes | 1 | Compromise: smaller footprint, slight CPU cost |
| `DISK_ONLY` | ❌ | ✅ | Yes | 1 | Very large datasets, fast disks |
| `OFF_HEAP` | ✅ | ❌ | Yes | 1 | Avoid GC pressure (Tungsten) |

In [ ]:
# MEMORY_ONLY — fastest access, partition lost if memory pressure
df_mem = df.persist(StorageLevel.MEMORY_ONLY)
df_mem.count()  # trigger
df_mem.show()

# DISK_ONLY — useful for very large intermediate results
df_disk = enriched.persist(StorageLevel.DISK_ONLY)
df_disk.count()  # trigger
print("Disk-persisted count:", df_disk.count())

## 3. unpersist() — Release Cached Data

Cached data occupies executor memory until:
1. You call `unpersist()`
2. LRU eviction when memory is needed
3. The SparkSession ends

> **Best practice:** Always `unpersist()` when you're done with a cached DataFrame in a long-running notebook or job.

In [ ]:
# Release all cached DataFrames
enriched.unpersist()
df_mem.unpersist()
df_disk.unpersist()

print("All caches released")

## 4. Verifying Cache with explain()

After caching and triggering an action, `explain()` shows `InMemoryTableScan` or `InMemoryRelation` in the physical plan — confirming the cache is being read.

In [ ]:
cached = df.cache()
cached.count()  # materialise

# Physical plan should show InMemoryTableScan
cached.filter(col("salary") > 80000).explain()

cached.unpersist()

## 5. When Caching Helps vs Hurts

### Cache when:
- The same DataFrame is used in **multiple downstream actions** (count, show, write, join)
- Recomputation is expensive (large joins, complex aggregations, reading from object storage)
- Iterative algorithms (ML model training)

### Don't cache when:
- The DataFrame is used **only once** — caching adds overhead with no benefit
- The dataset is **larger than available memory** and spilling to disk is slower than recomputing
- The source is fast (e.g., cached Delta Lake with data skipping — already optimised)
- AQE or predicate pushdown would eliminate most data before your code runs

In [ ]:
# Good cache use case: shared base after expensive join
base = df.join(broadcast(dept_info), on="dept").cache()
base.count()  # materialise once

# Three downstream consumers — all read from cache
q1 = base.groupBy("division").agg(count("*").alias("n"))
q2 = base.groupBy("dept").agg(round(avg("salary"), 0).alias("avg_salary"))
q3 = base.filter(col("country") == "RO")

q1.show()
q2.show()
q3.show()

base.unpersist()

## 6. createOrReplaceTempView vs Cache

Temp views are **logical aliases** — they don't cache data. The underlying query re-runs every time the view is queried.

To cache a temp view's result, persist the DataFrame first, then create the view.

In [ ]:
# Cache the DataFrame, then expose via SQL view
enriched_cached = df.join(broadcast(dept_info), on="dept").cache()
enriched_cached.count()  # materialise
enriched_cached.createOrReplaceTempView("enriched")

# SQL queries now read from cache
spark.sql("SELECT division, AVG(salary) FROM enriched GROUP BY division").show()
spark.sql("SELECT country, COUNT(*) FROM enriched GROUP BY country").show()

enriched_cached.unpersist()

---

## Day 14 Checklist

- [ ] Used `cache()` — know it maps to `MEMORY_AND_DISK`
- [ ] Used `persist()` with a specific `StorageLevel`
- [ ] Know all 5 storage levels and when to choose each
- [ ] Confirmed `InMemoryTableScan` in `explain()` output
- [ ] Called `unpersist()` when done with cached data
- [ ] Know when caching hurts (single use, dataset larger than RAM)
- [ ] Know that temp views don't cache data

**Next:** Day 15 — Batch Pipeline Project (end-to-end ETL with medallion architecture)

---

## Week 2 Complete!

| Day | Topic | Exam Domain |
|---|---|---|
| 8  | Aggregations & GroupBy | DataFrame API 30% |
| 9  | Joins (all 7 types) | DataFrame API 30% |
| 10 | String & Date Functions | DataFrame API 30% |
| 11 | Math & Collection Functions | DataFrame API 30% |
| 12 | UDFs & Pandas UDFs | DataFrame API 30% |
| 13 | Window Functions | DataFrame API 30% |
| 14 | Caching & Persistence | Optimisation 10% + Architecture 20% |